# Huấn Luyện Mô Hình Nhận Diện Mệnh Giá Tiền Việt Nam (VND) Cho Ứng Dụng Mắt Việt
### Dự án: Mắt Việt (VisionAssist) - Trợ lý thị giác cho người khiếm thị

> **Mục tiêu:** Xây dựng mô hình phân loại tiền tệ polymer/cotton Việt Nam và phát hiện đối tượng không phải tiền (lớp `none`) dựa trên kiến trúc **MobileNetV3-Large**.
> Tối ưu hóa cho thiết bị di động Android chạy **100% On-Device Offline** với TensorFlow Lite (lượng tử hóa FP16).

---
### Quy trình thực hiện trong Notebook:
1. **Cấu hình & Nạp dữ liệu tiền VND:** Hỗ trợ nạp dataset từ Google Drive (thư mục/file zip) hoặc Kaggle API.
2. **Tải & Chuẩn bị lớp 'none' (COCO val2017):** Tải ảnh từ COCO val2017 (1200 ảnh), cắt cận cảnh ngẫu nhiên 50% ảnh (30-70% kích thước) và thu nhỏ cạnh dài tối đa 640 để mô phỏng vật thể trong đời sống.
3. **Dọn dẹp & Tự động trích xuất nhãn:** Đọc danh sách lớp tự động từ cấu trúc thư mục (bao gồm cả lớp `none`), chuẩn hóa EXIF, kiểm tra ảnh hỏng.
4. **Chống rò rỉ dữ liệu & Kiểm tra tỷ lệ lớp 'none':** Dùng Perceptual Hash (`imagehash`) gom nhóm ảnh tương đồng, phân chia 70% Train / 15% Val / 15% Test. Kiểm tra tỷ lệ lớp `none` (cảnh báo nếu > 20% hoặc < 5%).
5. **Pipeline & Data Augmentation (Không lật ảnh):** Tăng cường ảnh mô phỏng thực tế (xoay, zoom, đổi màu/sáng, làm mờ, nén JPEG, random erasing). Tuyệt đối **không lật ảnh** để bảo toàn chữ số mệnh giá.
6. **Huấn luyện 2 giai đoạn (Two-Phase Transfer Learning):** Giai đoạn 1 huấn luyện Classification Head ($lr = 10^{-3}$), Giai đoạn 2 fine-tuning mở khóa 30 lớp cuối ($lr = 10^{-5}$) giữ BatchNorm đóng băng.
7. **Đánh giá toàn diện & Báo cáo riêng cho lớp 'none':** Đánh giá Precision, Recall, Confusion Matrix, tỉ lệ tiền bị đoán nhầm thành none (từ chối tiền thật), và tỉ lệ none bị đoán nhầm thành tiền (đọc bừa).
8. **Xuất mô hình TFLite (FP16):** Lượng tử hóa FP16, xuất file nhãn `currency_labels.txt` (có `none`), từ điển đọc `label_display_vi.json` (`none` là chuỗi rỗng), và `metrics.json`.

In [ ]:
# @title Bước 1: Cài đặt thư viện phụ trợ và kiểm tra môi trường phần cứng
!pip install -q imagehash kaggle

import os
import sys
import json
import time
import io
import shutil
import random
import zipfile
import hashlib
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageOps, ImageFilter

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV3Large
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

try:
    import imagehash
    HAS_IMAGEHASH = True
except ImportError:
    HAS_IMAGEHASH = False
    print('[WARNING] Khong the nap thu vien imagehash, he thong se dung MD5 fallback.')

# Co dinh seed cho tinh tai lap thuc nghiem
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

print('=' * 60)
print('KIEM TRA THONG TIN HE THONG:')
print(f'- Python:      {sys.version.split()[0]}')
print(f'- TensorFlow:  {tf.__version__}')
print(f'- Keras:       {keras.__version__}')
if HAS_IMAGEHASH:
    print(f'- ImageHash:   {imagehash.__version__}')

# Kiem tra GPU phan cung
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'[INFO] GPU da san sang: {len(gpus)} thiet bi ({gpus[0].name})')
    !nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader
else:
    print('[WARNING] Chua bat GPU. Hay vao menu Runtime -> Change runtime type -> Chon T4 GPU de tang toc!')
print('=' * 60)

## Bước 2: Thiết lập cấu hình đầu vào & Tham số huấn luyện
Người dùng chỉ cần điều chỉnh các biến ở ô mã dưới đây nếu muốn đổi đường dẫn dữ liệu hoặc bật chế độ kiểm tra nhanh.

In [ ]:
# @title Bang cau hinh chinh (Nguoi dung chinh sua o nay theo nhu cau)

# 1. NGUON DU LIEU GOC (KAGGLE / VND / MONEY):
#  'drive_folder': Dung truc tiep thu muc 'VND' hoac 'money' tren Google Drive (Khuyen dung)
#  'drive_zip'   : Giai nen file zip tu Google Drive (vd /content/drive/MyDrive/VND.zip)
#  'auto'        : Tu dong tim thu muc 'VND' / 'money' hoac file zip tren Google Drive
#  'kaggle_api'  : Tai dataset tu Kaggle API (can file kaggle.json)
DATA_SOURCE = 'drive_folder'
DRIVE_PATH = '/content/drive/MyDrive/VND'

# 2. SO LUONG ANH CHO LOP 'none' TU COCO VAL2017:
COCO_SAMPLE_COUNT = 1200

# 3. CHE DO KIEM TRA NHANH (Smoke test):
#  True: Chay 1 epoch moi giai doan de kiem tra pipeline trong 1-2 phut
#  False: Huan luyen day du de dat do chinh xac toi uu (mac dinh)
QUICK_RUN = False

# 4. THAM SO HUAN LUYEN:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS_PHASE1 = 1 if QUICK_RUN else 12
EPOCHS_PHASE2 = 1 if QUICK_RUN else 12

# 5. THU MUC XUAT KET QUA:
OUTPUT_DIR = '/content/outputs'
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/vision_assist/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('[INFO] Da nap cau hinh:')
print(f'  - Nguon du lieu goc:        {DATA_SOURCE} ({DRIVE_PATH})')
print(f'  - So anh lop none tu COCO:  {COCO_SAMPLE_COUNT}')
print(f'  - Che do QUICK_RUN:         {QUICK_RUN}')
print(f'  - Kich thuoc anh:           {IMG_SIZE}')
print(f'  - Thu muc luu ket qua:      {OUTPUT_DIR}')

## Bước 3: Nạp và giải nén dữ liệu tiền VND vào máy ảo Colab
Hệ thống sẽ lấy dữ liệu từ Google Drive hoặc tải dataset từ Kaggle và sao chép vào ổ SSD máy ảo để đọc với tốc độ cao.

In [ ]:
# @title Thuc thi nap du lieu tien VND vao may ao Colab

RAW_DATA_DIR = '/content/data/raw'
os.makedirs(RAW_DATA_DIR, exist_ok=True)

def mount_drive_if_needed():
    """Gan ket Google Drive neu chua co"""
    if not os.path.exists('/content/drive/MyDrive'):
        print('[INFO] Dang gan ket Google Drive...')
        from google.colab import drive
        drive.mount('/content/drive')
        print('[SUCCESS] Da gan ket Google Drive thanh cong.')
    else:
        print('[INFO] Google Drive da duoc gan ket san.')

def find_dataset_folder_on_drive(preferred_path):
    """Tu dong tim thu muc VND hoac money tren Drive"""
    if os.path.exists(preferred_path) and os.path.isdir(preferred_path):
        return preferred_path
    drive_root = '/content/drive/MyDrive'
    if not os.path.exists(drive_root):
        return None
    common_candidates = [
        os.path.join(drive_root, 'VND'),
        os.path.join(drive_root, 'vnd'),
        os.path.join(drive_root, 'money'),
        os.path.join(drive_root, 'dataset', 'VND'),
        os.path.join(drive_root, 'dataset', 'money'),
        os.path.join(drive_root, 'data', 'VND'),
        os.path.join(drive_root, 'data', 'money'),
    ]
    for p in common_candidates:
        if os.path.exists(p) and os.path.isdir(p):
            return p
    KNOWN_DENOMS = {'1k', '2k', '5k', '10k', '20k', '50k', '100k', '200k', '500k', '1000', '2000', '5000', '10000', '20000', '50000', '100000', '200000', '500000'}
    for root, dirs, _ in os.walk(drive_root):
        depth = root[len(drive_root):].count(os.sep)
        if depth > 3:
            continue
        for d in dirs:
            if d.lower() in ('vnd', 'money'):
                candidate = os.path.join(root, d)
                try:
                    subdirs = {s.lower() for s in os.listdir(candidate) if os.path.isdir(os.path.join(candidate, s))}
                    if subdirs.intersection(KNOWN_DENOMS) or ('train' in subdirs):
                        return candidate
                except Exception:
                    pass
    return None

def download_from_kaggle(target_dir):
    """Tai dataset tu Kaggle API khi co file credentials"""
    print('[INFO] Dang chuan bi tai dataset tu Kaggle API...')
    kaggle_json_path = os.path.expanduser('~/.kaggle/kaggle.json')
    if not os.path.exists(kaggle_json_path):
        print('[INFO] Vui long tai len file kaggle.json cua ban:')
        from google.colab import files
        uploaded = files.upload()
        if 'kaggle.json' in uploaded:
            os.makedirs(os.path.dirname(kaggle_json_path), exist_ok=True)
            with open(kaggle_json_path, 'wb') as f:
                f.write(uploaded['kaggle.json'])
            os.chmod(kaggle_json_path, 0o600)
            print('[SUCCESS] Da luu cau hinh Kaggle API.')
        else:
            raise FileNotFoundError('Khong tim thay file kaggle.json.')
    cmd = f'kaggle datasets download -d nguyentrongdai/vietnamese-currency -p {target_dir} --unzip'
    ret = os.system(cmd)
    if ret != 0:
        raise RuntimeError('Tai dataset tu Kaggle that bai.')
    print('[SUCCESS] Tai va giai nen Kaggle hoan tat.')

actual_mode = DATA_SOURCE
if DATA_SOURCE == 'auto':
    try:
        mount_drive_if_needed()
        found_folder = find_dataset_folder_on_drive(DRIVE_PATH)
        if found_folder:
            actual_mode = 'drive_folder'
            DRIVE_PATH = found_folder
        elif os.path.exists(DRIVE_PATH) and DRIVE_PATH.endswith('.zip'):
            actual_mode = 'drive_zip'
        else:
            actual_mode = 'kaggle_api'
    except Exception:
        actual_mode = 'kaggle_api'

print(f'[INFO] Nguon du lieu duoc kich hoat: [{actual_mode}]')
dest_dataset_dir = os.path.join(RAW_DATA_DIR, 'dataset')

if actual_mode == 'drive_folder':
    mount_drive_if_needed()
    folder_to_use = find_dataset_folder_on_drive(DRIVE_PATH)
    if not folder_to_use:
        raise FileNotFoundError(f'Khong tim thay thu muc tai {DRIVE_PATH}.')
    print(f'[INFO] Sao chep tu Drive ({folder_to_use}) sang Colab SSD ({dest_dataset_dir})...')
    shutil.copytree(folder_to_use, dest_dataset_dir, dirs_exist_ok=True)
    print('[SUCCESS] Sao chep du lieu VND hoan tat.')
elif actual_mode == 'drive_zip':
    mount_drive_if_needed()
    if not os.path.exists(DRIVE_PATH):
        raise FileNotFoundError(f'Khong tim thay file zip: {DRIVE_PATH}')
    print(f'[INFO] Giai nen file zip: {DRIVE_PATH}...')
    with zipfile.ZipFile(DRIVE_PATH, 'r') as zip_ref:
        zip_ref.extractall(dest_dataset_dir)
    print('[SUCCESS] Giai nen file zip hoan tat.')
elif actual_mode == 'kaggle_api':
    download_from_kaggle(dest_dataset_dir)

## Bước 4: Tải COCO val2017 và tạo lớp 'none' (không phải tiền)
- **Mục đích:** Bổ sung lớp nhận diện âm bản (negative class) giúp ứng dụng Mắt Việt phát hiện khi người dùng chĩa camera vào đồ vật, mặt bàn, người... chứ không có tiền trong khung hình.
- **Quy trình:**
  1. Tải tập ảnh `COCO val2017` (`http://images.cocodataset.org/zips/val2017.zip`).
  2. Lấy ngẫu nhiên $N = 1200$ ảnh (`COCO_SAMPLE_COUNT = 1200`).
  3. **Một nửa số ảnh ($600$ ảnh) được cắt cận cảnh ngẫu nhiên (30% - 70% kích thước)** nhằm mô phỏng góc chụp sát camera của người khiếm thị.
  4. Thu nhỏ cạnh dài tối đa **640 pixel** (`img.thumbnail((640, 640))`).
  5. Lưu vào thư mục `<DATA_ROOT>/none` (cùng cấp với các thư mục mệnh giá tiền).
  6. **Tối ưu hóa:** Bỏ qua tải nếu thư mục `none` đã có đủ ảnh; tự động xóa file `.zip` sau khi trích xuất xong để tiết kiệm đĩa cứng Colab.

In [ ]:
# @title Thuc thi tai COCO val2017 va chuan bi lop 'none'

N_IMAGES = COCO_SAMPLE_COUNT

# Tim thu muc goc chua cac lop menh gia trong RAW_DATA_DIR
def find_currency_data_root(base_dir):
    KNOWN_DENOMS = {'1k', '2k', '5k', '10k', '20k', '50k', '100k', '200k', '500k', 
                    '1000', '2000', '5000', '10000', '20000', '50000', '100000', '200000', '500000'}
    for root, dirs, _ in os.walk(base_dir):
        matched = set(d.lower() for d in dirs).intersection(KNOWN_DENOMS)
        if len(matched) >= 3:
            return root
    return os.path.join(base_dir, 'dataset')

currency_data_root = find_currency_data_root(RAW_DATA_DIR)
none_dir = os.path.join(currency_data_root, 'none')
os.makedirs(none_dir, exist_ok=True)

existing_none_images = [f for f in os.listdir(none_dir) if os.path.splitext(f)[1].lower() in {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}]

if len(existing_none_images) >= N_IMAGES:
    print(f'[INFO] Thu muc none da co san {len(existing_none_images)} anh (>= {N_IMAGES} anh). Bo qua buoc tai COCO.')
else:
    print(f'[INFO] Thu muc none hien co {len(existing_none_images)} anh. Tien hanh tai va tao {N_IMAGES} anh tu COCO val2017 vao: {none_dir}...')
    coco_zip_path = '/content/val2017.zip'
    if not os.path.exists(coco_zip_path):
        print('[INFO] Dang tai COCO val2017 (http://images.cocodataset.org/zips/val2017.zip)...')
        ret = os.system('wget -q -c http://images.cocodataset.org/zips/val2017.zip -O /content/val2017.zip')
        if ret != 0 or not os.path.exists(coco_zip_path):
            import urllib.request
            print('[INFO] Thu tai lai bang urllib...')
            urllib.request.urlretrieve('http://images.cocodataset.org/zips/val2017.zip', coco_zip_path)
    
    print(f'[INFO] Dang trich xuat ngau nhien {N_IMAGES} anh va ap dung cat can canh 50% so anh...')
    with zipfile.ZipFile(coco_zip_path, 'r') as zip_ref:
        img_members = [m for m in zip_ref.namelist() if m.lower().endswith(('.jpg', '.jpeg', '.png'))]
        selected_members = random.sample(img_members, min(N_IMAGES, len(img_members)))
        
        half_crop_count = len(selected_members) // 2
        for idx, m_name in enumerate(selected_members):
            if (idx + 1) % 300 == 0 or (idx + 1) == len(selected_members):
                print(f'  - Tien do chuan bi anh none: {idx + 1}/{len(selected_members)}...')
            
            im_bytes = zip_ref.read(m_name)
            try:
                with Image.open(io.BytesIO(im_bytes)) as img:
                    img = ImageOps.exif_transpose(img).convert('RGB')
                    w, h = img.size
                    
                    # 1. Mot nua so anh duoc cat can canh ngau nhien (30-70% kich thuoc)
                    if idx < half_crop_count:
                        crop_ratio = random.uniform(0.30, 0.70)
                        crop_w = max(20, int(w * crop_ratio))
                        crop_h = max(20, int(h * crop_ratio))
                        left = random.randint(0, max(0, w - crop_w))
                        top = random.randint(0, max(0, h - crop_h))
                        img = img.crop((left, top, left + crop_w, top + crop_h))
                    
                    # 2. Thu nho canh dai toi da 640
                    img.thumbnail((640, 640), Image.Resampling.LANCZOS)
                    
                    dst_path = os.path.join(none_dir, f'coco_none_{idx:05d}.jpg')
                    img.save(dst_path, format='JPEG', quality=90)
            except Exception:
                pass

    final_none_count = len([f for f in os.listdir(none_dir) if os.path.splitext(f)[1].lower() in {'.jpg', '.jpeg', '.png'}])
    print(f'[SUCCESS] Da luu {final_none_count} anh vao thu muc none tai: {none_dir}')
    
    # Xoa file zip de giai phong o dia Colab
    if os.path.exists(coco_zip_path):
        os.remove(coco_zip_path)
        print('[INFO] Da xoa file val2017.zip de tiet kiem dung luong dia.')

## Bước 5: Tự động dò tìm thư mục gốc, gộp dữ liệu & kiểm tra ảnh hỏng
- Quét đệ quy tìm tất cả ảnh hợp lệ (`.jpg`, `.jpeg`, `.png`, `.webp`, `.bmp`).
- Gom toàn bộ ảnh về theo nhãn lớp (bao gồm cả lớp `none` và các mệnh giá) vào `/content/consolidated_data` để chuẩn bị chia lại từ đầu nhằm kiểm soát triệt để rò rỉ dữ liệu.
- Áp dụng `ImageOps.exif_transpose` chuẩn hóa góc xoay và chuyển sang không gian màu `RGB`.
- Tự động đọc danh sách nhãn lớp (`class_names`) sắp xếp theo bảng chữ cái từ tên thư mục, **tuyệt đối không hard-code**.

In [ ]:
# @title Quet va gom anh ve thu muc chuan hoa /content/consolidated_data

VALID_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
CONSOLIDATED_DIR = '/content/consolidated_data'
if os.path.exists(CONSOLIDATED_DIR):
    shutil.rmtree(CONSOLIDATED_DIR)
os.makedirs(CONSOLIDATED_DIR, exist_ok=True)

print('[INFO] Dang duyet cay thu muc du lieu tho...')
all_files = []
for root, dirs, files in os.walk(RAW_DATA_DIR):
    for file in files:
        ext = os.path.splitext(file)[1].lower()
        if ext in VALID_EXTS:
            all_files.append(os.path.join(root, file))

print(f'[INFO] Tim thay tong cong {len(all_files)} tep anh.')
if len(all_files) == 0:
    raise FileNotFoundError(f'Khong tim thay anh nao trong {RAW_DATA_DIR}!')

# Thu muc ky thuat can bo qua khi trich xuat ten lop
IGNORE_DIRS = {'train', 'test', 'val', 'valid', 'validation', 'raw', 'data', 'images', 'money', 'vnd', 'dataset', 'vietnamese-currency'}

def extract_class_name(file_path):
    parts = Path(file_path).parts
    for part in reversed(parts[:-1]):
        clean_part = part.lower().strip()
        if clean_part not in IGNORE_DIRS and not clean_part.startswith('.'):
            return part.strip()
    return Path(file_path).parent.name.strip()

corrupt_files = []
valid_samples = []

print('[INFO] Dang kiem tra tinh hop le cua anh va gom ve thu muc thong nhat...')
for idx, file_path in enumerate(all_files):
    if (idx + 1) % 2000 == 0 or (idx + 1) == len(all_files):
        print(f'   - Tien do xu ly: {idx + 1}/{len(all_files)} anh...')
    class_name = extract_class_name(file_path)
    class_dir = os.path.join(CONSOLIDATED_DIR, class_name)
    os.makedirs(class_dir, exist_ok=True)
    
    try:
        with Image.open(file_path) as img:
            img = ImageOps.exif_transpose(img)
            img = img.convert('RGB')
            dest_filename = f'{class_name}_{idx:06d}.jpg'
            dest_path = os.path.join(class_dir, dest_filename)
            img.save(dest_path, format='JPEG', quality=95)
            valid_samples.append((dest_path, class_name))
    except Exception as e:
        corrupt_files.append((file_path, str(e)))

print(f'[SUCCESS] Da xu ly xong {len(valid_samples)} anh hop le.')
if corrupt_files:
    print(f'[WARNING] Phat hien {len(corrupt_files)} anh bi loi/hong (da bo qua).')

# Tu dong trich xuat danh sach lop
class_names = sorted([d for d in os.listdir(CONSOLIDATED_DIR) if os.path.isdir(os.path.join(CONSOLIDATED_DIR, d))])
num_classes = len(class_names)

print('=' * 60)
print(f'DANH SACH {num_classes} LOP TIM THAY TU DU LIEU (TU DONG):')
class_counts = {}
for cname in class_names:
    count = len(os.listdir(os.path.join(CONSOLIDATED_DIR, cname)))
    class_counts[cname] = count
    print(f'  - Lop [{cname:6s}]: {count} anh')
print(f'Tong so anh hop le: {sum(class_counts.values())}')
print('=' * 60)

# Bieu do cot so anh moi lop
plt.figure(figsize=(12, 4.5))
df_counts = pd.DataFrame(list(class_counts.items()), columns=['Lop', 'So luong'])
palette = sns.color_palette('viridis', len(df_counts))
bars = plt.bar(df_counts['Lop'], df_counts['So luong'], color=palette, edgecolor='black', alpha=0.85)
plt.title('Phan Bo So Luong Anh Theo Lop (Bao Gom Lop none)', fontsize=12, fontweight='bold', pad=12)
plt.xlabel('Lop (Class)', fontsize=11)
plt.ylabel('So luong anh', fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.5)
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 2, f'{int(yval)}', ha='center', va='bottom', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.show()

## Bước 6: Phân chia tập dữ liệu chống rò rỉ (Deduplication & Stratified Grouping)
- **Chống rò rỉ dữ liệu (Anti-Leakage Splitting):**
  1. Tính mã băm tri giác **Perceptual Hash (`imagehash.phash`)** cho toàn bộ ảnh.
  2. Gom tất cả các ảnh có khoảng cách Hamming $\le 4$ vào cùng một nhóm duy nhất (dùng thuật toán Disjoint Set / Union-Find).
  3. Phân chia theo tỷ lệ **70% Train / 15% Val / 15% Test** theo từng lớp bảo toàn tỷ lệ nhóm độc lập.
- **Báo cáo và kiểm tra tỷ lệ lớp 'none':**
  - Báo số ảnh của lớp `none` ở từng tập (Train, Val, Test).
  - Cảnh báo nếu lớp `none` chiếm **quá 20%** hoặc **ít hơn 5%** tổng số ảnh toàn bộ dataset.

In [ ]:
# @title Gom nhom anh tuong dong va chia 70% Train / 15% Val / 15% Test

class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
    def find(self, i):
        if self.parent[i] == i:
            return i
        self.parent[i] = self.find(self.parent[i])
        return self.parent[i]
    def union(self, i, j):
        root_i = self.find(i)
        root_j = self.find(j)
        if root_i != root_j:
            self.parent[root_i] = root_j

dataset_records = []
for cname in class_names:
    cdir = os.path.join(CONSOLIDATED_DIR, cname)
    for fname in sorted(os.listdir(cdir)):
        fpath = os.path.join(cdir, fname)
        dataset_records.append({'path': fpath, 'label': cname})

total_imgs = len(dataset_records)
print(f'[INFO] Dang tinh ma bam tri giac (Perceptual Hash) cho {total_imgs} anh...')

uf = UnionFind(total_imgs)
hashes = []

if HAS_IMAGEHASH:
    for idx, r in enumerate(dataset_records):
        if (idx + 1) % 2000 == 0 or (idx + 1) == total_imgs:
            print(f'   - Dang tinh hash: {idx + 1}/{total_imgs} anh...')
        try:
            with Image.open(r['path']) as im:
                h = imagehash.phash(im)
                hashes.append(h)
        except Exception:
            hashes.append(None)
    
    # Gom nhom theo khoang cach Hamming <= 4 (chi so sanh giua cac anh trong cung nhan lop)
    label_to_indices = {}
    for idx, r in enumerate(dataset_records):
        label_to_indices.setdefault(r['label'], []).append(idx)
    
    duplicate_count = 0
    for lbl, indices in label_to_indices.items():
        n_idx = len(indices)
        for i in range(n_idx):
            idx_i = indices[i]
            hash_i = hashes[idx_i]
            if hash_i is None: continue
            for j in range(i + 1, n_idx):
                idx_j = indices[j]
                hash_j = hashes[idx_j]
                if hash_j is None: continue
                if hash_i - hash_j <= 4:
                    uf.union(idx_i, idx_j)
                    duplicate_count += 1
    print(f'[SUCCESS] Hoan tat gom nhom tri giac (phat hien {duplicate_count} cap anh gan trung).')
else:
    print('[WARNING] imagehash khong kha dung, dung MD5 fallback...')
    md5_map = {}
    for idx, r in enumerate(dataset_records):
        with open(r['path'], 'rb') as f:
            digest = hashlib.md5(f.read()).hexdigest()
        if digest in md5_map:
            uf.union(idx, md5_map[digest])
        else:
            md5_map[digest] = idx

for idx in range(total_imgs):
    dataset_records[idx]['group_id'] = uf.find(idx)

df_samples = pd.DataFrame(dataset_records)
num_unique_groups = df_samples['group_id'].nunique()
print(f'[INFO] {total_imgs} anh duoc gom thanh {num_unique_groups} nhom doc lap.')

# Phan chia 70% Train, 15% Val, 15% Test
train_paths, val_paths, test_paths = [], [], []

for cname in class_names:
    df_c = df_samples[df_samples['label'] == cname]
    group_sizes = df_c.groupby('group_id').size().to_dict()
    groups = list(group_sizes.keys())
    random.shuffle(groups)
    
    total_class_imgs = len(df_c)
    target_train = int(total_class_imgs * 0.70)
    target_val = int(total_class_imgs * 0.15)
    
    curr_train, curr_val = 0, 0
    train_grps, val_grps, test_grps = set(), set(), set()
    
    for grp in groups:
        sz = group_sizes[grp]
        if curr_train + sz <= target_train or (curr_train == 0 and len(train_grps) == 0):
            train_grps.add(grp)
            curr_train += sz
        elif curr_val + sz <= target_val or (curr_val == 0 and len(val_grps) == 0):
            val_grps.add(grp)
            curr_val += sz
        else:
            test_grps.add(grp)
    
    train_paths.extend(df_c[df_c['group_id'].isin(train_grps)]['path'].tolist())
    val_paths.extend(df_c[df_c['group_id'].isin(val_grps)]['path'].tolist())
    test_paths.extend(df_c[df_c['group_id'].isin(test_grps)]['path'].tolist())

# Kiem tra va bao cao so anh lop 'none' o tung tap
none_train_count = sum(1 for p in train_paths if Path(p).parent.name == 'none')
none_val_count   = sum(1 for p in val_paths if Path(p).parent.name == 'none')
none_test_count  = sum(1 for p in test_paths if Path(p).parent.name == 'none')
none_total       = none_train_count + none_val_count + none_test_count
none_percentage  = (none_total / total_imgs) * 100.0 if total_imgs > 0 else 0.0

print('=' * 60)
print('THONG KE SO LUONG ANH LOP "NONE" (KHONG PHAI TIEN):')
print(f'  - Tap Train (70%): {none_train_count} anh')
print(f'  - Tap Val   (15%): {none_val_count} anh')
print(f'  - Tap Test  (15%): {none_test_count} anh')
print(f'  - Tong cong:       {none_total} / {total_imgs} anh ({none_percentage:.2f}%)')

if none_percentage > 20.0:
    print(f'[CANH BAO] Lop "none" chiem {none_percentage:.2f}% tong so anh (> 20%). Ty le nay qua lon co the khien mo hinh thien vi du doan khong phai tien!')
elif none_percentage < 5.0:
    print(f'[CANH BAO] Lop "none" chi chiem {none_percentage:.2f}% tong so anh (< 5%). Ty le nay qua it co the khien mo hinh bo sot cac vat the ngoai doi!')
else:
    print(f'[INFO] Ty le lop "none" dat muc can bang ly tuong: {none_percentage:.2f}% (trong nguong an toan 5% - 20%).')
print('=' * 60)

# Bang thong ke tong hop phan chia
summary_rows = []
for cname in class_names:
    n_tr = sum(1 for p in train_paths if Path(p).parent.name == cname)
    n_va = sum(1 for p in val_paths if Path(p).parent.name == cname)
    n_te = sum(1 for p in test_paths if Path(p).parent.name == cname)
    summary_rows.append({'Lop': cname, 'Train (70%)': n_tr, 'Val (15%)': n_va, 'Test (15%)': n_te, 'Tong': n_tr + n_va + n_te})

df_split_summary = pd.DataFrame(summary_rows)
print(df_split_summary.to_string(index=False))
print(f'Tong cong: Train = {len(train_paths)} | Val = {len(val_paths)} | Test = {len(test_paths)}')

## Bước 7: Xây dựng `tf.data` Pipeline & Tăng cường ảnh mạnh (Strong Augmentation)
- **Quy tắc quan trọng:** Mô hình nhận pixel thô $[0, 255]$ kiểu `float32` nhờ tính năng `include_preprocessing=True` của MobileNetV3. Do đó **KHÔNG chia 255 bên ngoài**.
- **Augmentation mạnh mẽ trong tf.data (không nhúng trong model):**
  - Xoay ngẫu nhiên $\pm 10^\circ$ đến $\pm 15^\circ$
  - Thu phóng ngẫu nhiên $\pm 15\%$
  - Dịch chuyển nhẹ $\pm 5\%$
  - Thay đổi độ sáng ngẫu nhiên $\pm 20\%$ và tương phản $\pm 20\%$
  - Thêm nhiễu Gaussian nhẹ mô phỏng camera nhiễu ISO
  - **TUYỆT ĐỐI KHÔNG lật ngang (Horizontal Flip) hoặc lật dọc (Vertical Flip):** Tiền giấy có chữ và số định danh, nếu lật ngược chữ số sẽ làm sai lệch đặc trưng nhận diện.
- Tối ưu luồng I/O với `.shuffle()`, `.batch()`, và `.prefetch(tf.data.AUTOTUNE)`.

In [ ]:
# @title Xay dung tf.data Pipeline va kiem tra lop tang cuong du lieu

class_to_idx = {name: idx for idx, name in enumerate(class_names)}

data_augmentation_layers = keras.Sequential([
    layers.RandomRotation(factor=(-0.035, 0.035), fill_mode='nearest', seed=RANDOM_SEED),
    layers.RandomZoom(height_factor=(-0.15, 0.15), fill_mode='nearest', seed=RANDOM_SEED),
    layers.RandomTranslation(height_factor=(-0.08, 0.08), width_factor=(-0.08, 0.08), fill_mode='nearest', seed=RANDOM_SEED),
    layers.RandomBrightness(factor=0.25, value_range=(0, 255), seed=RANDOM_SEED),
    layers.RandomContrast(factor=0.25, seed=RANDOM_SEED),
], name='data_augmentation')

def load_and_preprocess_image(path, label_idx):
    img_bytes = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img_bytes, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32)  # Giu nguyen gia tri tho 0-255
    return img, label_idx

def apply_train_augmentation(images, labels):
    augmented_images = data_augmentation_layers(images, training=True)
    noise = tf.random.normal(shape=tf.shape(augmented_images), mean=0.0, stddev=3.0, dtype=tf.float32)
    augmented_images = tf.clip_by_value(augmented_images + noise, 0.0, 255.0)
    return augmented_images, labels

def create_dataset(file_paths, is_training=False):
    paths = [p for p in file_paths]
    labels = [class_to_idx[Path(p).parent.name] for p in file_paths]
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if is_training:
        ds = ds.shuffle(buffer_size=len(paths), seed=RANDOM_SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    if is_training:
        ds = ds.map(apply_train_augmentation, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.prefetch(buffer_size=tf.data.AUTOTUNE)
    return ds

train_ds = create_dataset(train_paths, is_training=True)
val_ds   = create_dataset(val_paths, is_training=False)
test_ds  = create_dataset(test_paths, is_training=False)

print(f'[SUCCESS] Da khoi tao tf.data pipeline thanh cong (Batch Size = {BATCH_SIZE}).')

# Hien thi mau anh sau khi di qua Data Augmentation
sample_batch_imgs, sample_batch_lbls = next(iter(train_ds))
plt.figure(figsize=(12, 6))
for i in range(min(8, len(sample_batch_imgs))):
    plt.subplot(2, 4, i + 1)
    plt.imshow(tf.cast(sample_batch_imgs[i], tf.uint8).numpy())
    lbl_name = class_names[sample_batch_lbls[i].numpy()]
    plt.title(f'Augmented: {lbl_name}', fontsize=10)
    plt.axis('off')
plt.suptitle('Anh Mau Sau Khi Qua Pipeline Augmentation (Khong lat nguoc chu so)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## Bước 8: Khởi tạo mô hình MobileNetV3-Large & Tính Class Weights
- **Backbone:** `MobileNetV3Large(include_top=False, weights='imagenet', input_shape=(224, 224, 3), include_preprocessing=True)`.
- **Classification Head:** `GlobalAveragePooling2D()` $\rightarrow$ `Dropout(0.3)` $\rightarrow$ `Dense(num_classes, activation='softmax')`.
- **Class Weights:** Tính trọng số cân bằng lớp từ `sklearn.utils.class_weight.compute_class_weight` để bù trừ trường hợp dữ liệu chênh lệch giữa các mệnh giá và lớp `none`.

In [ ]:
# @title Xay dung kien truc mo hinh MobileNetV3Large va tinh Class Weights

train_labels_list = [class_to_idx[Path(p).parent.name] for p in train_paths]
computed_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels_list),
    y=np.array(train_labels_list)
)
class_weight_dict = {int(cls_i): float(w) for cls_i, w in zip(np.unique(train_labels_list), computed_weights)}
print('[INFO] Class Weights duoc ap dung:')
for idx, cname in enumerate(class_names):
    print(f'  - Lop [{cname:6s}]: {class_weight_dict.get(idx, 1.0):.3f}')

def build_currency_model(num_classes):
    backbone = MobileNetV3Large(
        input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3),
        include_top=False,
        weights='imagenet',
        include_preprocessing=True
    )
    backbone.trainable = False
    
    inputs = keras.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3), dtype=tf.float32, name='input_image')
    x = backbone(inputs)
    x = layers.GlobalAveragePooling2D(name='global_avg_pool')(x)
    x = layers.Dropout(0.3, name='head_dropout')(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='prediction_softmax')(x)
    
    model = keras.Model(inputs=inputs, outputs=outputs, name='vnd_currency_mobilenetv3')
    return model, backbone

model, backbone = build_currency_model(num_classes)
model.summary()

## Bước 9: Huấn luyện mô hình hai giai đoạn (Two-Phase Transfer Learning)
1. **Giai đoạn 1 (Feature Extraction):** Đóng băng toàn bộ backbone, chỉ huấn luyện Classification Head với tốc độ học $lr = 10^{-3}$.
2. **Giai đoạn 2 (Fine-Tuning):** Mở khóa khoảng 30 lớp cuối của backbone với tốc độ học nhỏ $lr = 10^{-5}$.
   - **Điểm kỹ thuật quan trọng:** Giữ toàn bộ các lớp `BatchNormalization` ở trạng thái **đóng băng (`trainable = False`)** để tránh làm trôi mean/variance của ImageNet.
3. **Callbacks:** `EarlyStopping(patience=3)` khôi phục trọng số tốt nhất, `ModelCheckpoint` lưu mô hình có `val_accuracy` cao nhất.

In [ ]:
# @title Huan luyen mo hinh Giai doan 1 (Head) va Giai doan 2 (Fine-Tuning)

CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, 'best_currency_model.keras')

# 1. Giai doan 1: Huan luyen Head
callbacks_p1 = [
    keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True, mode='max', verbose=1),
    keras.callbacks.ModelCheckpoint(filepath=CHECKPOINT_PATH, monitor='val_accuracy', save_best_only=True, mode='max', verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1)
]

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

print(f'[INFO] Bat dau Giai doan 1 (Toi da {EPOCHS_PHASE1} epochs)...')
start_p1 = time.time()
history_p1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_PHASE1,
    class_weight=class_weight_dict,
    callbacks=callbacks_p1,
    verbose=1
)
print(f'[SUCCESS] Giai doan 1 hoan tat trong {time.time() - start_p1:.1f}s.')

# 2. Giai doan 2: Mo khoa 30 lop cuoi (Giu BatchNorm dong bang)
backbone.trainable = True
NUM_FINE_TUNE = 30
for layer in backbone.layers[:-NUM_FINE_TUNE]:
    layer.trainable = False

for layer in backbone.layers[-NUM_FINE_TUNE:]:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False
    else:
        layer.trainable = True

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

callbacks_p2 = [
    keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True, mode='max', verbose=1),
    keras.callbacks.ModelCheckpoint(filepath=CHECKPOINT_PATH, monitor='val_accuracy', save_best_only=True, mode='max', verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-7, verbose=1)
]

initial_p2 = len(history_p1.history['accuracy'])
total_ep = initial_p2 + EPOCHS_PHASE2
print(f'[INFO] Bat dau Giai doan 2 Fine-Tuning (Epoch {initial_p2 + 1} -> {total_ep})...')
start_p2 = time.time()
history_p2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=total_ep,
    initial_epoch=initial_p2,
    class_weight=class_weight_dict,
    callbacks=callbacks_p2,
    verbose=1
)
print(f'[SUCCESS] Giai doan 2 hoan tat trong {time.time() - start_p2:.1f}s.')

# Ve do thi Loss va Accuracy
comb_acc = history_p1.history['accuracy'] + history_p2.history['accuracy']
comb_val_acc = history_p1.history['val_accuracy'] + history_p2.history['val_accuracy']
comb_loss = history_p1.history['loss'] + history_p2.history['loss']
comb_val_loss = history_p1.history['val_loss'] + history_p2.history['val_loss']

plt.figure(figsize=(12, 4.5))
plt.subplot(1, 2, 1)
plt.plot(comb_acc, label='Train Accuracy', color='#1f77b4', lw=2)
plt.plot(comb_val_acc, label='Val Accuracy', color='#ff7f0e', lw=2)
plt.axvline(x=initial_p2 - 0.5, color='gray', linestyle='--', label='Chuyen Fine-Tuning')
plt.title('Do Chinh Dac Qua Cac Epoch (Accuracy)', fontsize=11, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)

plt.subplot(1, 2, 2)
plt.plot(comb_loss, label='Train Loss', color='#1f77b4', lw=2)
plt.plot(comb_val_loss, label='Val Loss', color='#ff7f0e', lw=2)
plt.axvline(x=initial_p2 - 0.5, color='gray', linestyle='--', label='Chuyen Fine-Tuning')
plt.title('Ham Mat Mat Qua Cac Epoch (Loss)', fontsize=11, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)

curves_path = os.path.join(OUTPUT_DIR, 'training_curves.png')
plt.tight_layout()
plt.savefig(curves_path, dpi=300)
plt.show()
print(f'[INFO] Da luu do thi huan luyen vao: {curves_path}')

---
### Cảnh Báo Quan Trọng: Đánh Giá Tập Test & Hiệu Năng Thực Tế Ngoài Đời

> **1. Giới hạn của tập Test trên Benchmark:**
> - Tập Test hiện tại được chia tách từ cùng nguồn dataset (Roboflow/Kaggle) kết hợp với ảnh COCO val2017.
> - Mặc dù đã sử dụng mã băm tri giác (Perceptual Hash) để chống rò rỉ dữ liệu, kết quả Test Accuracy trên máy tính **chưa phản ánh 100% hiệu quả khi ứng dụng chạy ngoài đời thực**.

> **2. Điều kiện chụp thực tế của người khiếm thị:**
> - Trong đời sống hàng ngày, điều kiện chụp rất khắc nghiệt: tiền bị **nhăn nhúm, gập 2, gập 4, bị che một phần bởi ngón tay**, hoặc điều kiện ánh sáng nhập nhoạng ngoài chợ/đường phố.
> - Tương tự, các vật thể không phải tiền trong đời sống (bàn ghế, đồ vật, nền nhà, quần áo...) phong phú và phức tạp hơn nhiều so với tập ảnh mẫu COCO.

> **3. Yêu cầu kiểm thử bắt buộc:**
> - **Bắt buộc phải thử nghiệm thủ công (manual testing) trực tiếp trong app Mắt Việt:** Sau khi cài đặt file APK lên điện thoại Android, nhóm phát triển cần cầm tiền thật trên tay để thử nghiệm thực tế với nhiều khoảng cách, góc nghiêng và điều kiện ánh sáng khác nhau.
---

In [ ]:
# @title Danh gia tren tap TEST, ve Confusion Matrix va Bao cao rieng cho lop 'none'

# 1. Nap lai checkpoint tot nhat
if os.path.exists(CHECKPOINT_PATH):
    print(f'[INFO] Dang nap mo hinh tot nhat tu checkpoint: {CHECKPOINT_PATH}...')
    best_model = keras.models.load_model(CHECKPOINT_PATH)
else:
    best_model = model

# 2. Thu thap du doan tren tap TEST
y_true = []
y_pred_probs = []
for imgs, lbls in test_ds:
    probs = best_model.predict(imgs, verbose=0)
    y_pred_probs.append(probs)
    y_true.extend(lbls.numpy())

y_pred_probs = np.vstack(y_pred_probs)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.array(y_true)

test_acc = accuracy_score(y_true, y_pred)
print('=' * 60)
print(f'DO CHINH XAC TONG THE TREN TAP TEST: {test_acc * 100:.2f}%')
print('=' * 60)

# Classification Report tong the
report_str = classification_report(y_true, y_pred, target_names=class_names, digits=4, zero_division=0)
report_dict = classification_report(y_true, y_pred, target_names=class_names, output_dict=True, zero_division=0)
print(report_str)

# 3. Confusion Matrix chuan hoa theo hang
cm = confusion_matrix(y_true, y_pred, normalize='true')
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title(f'Normalized Confusion Matrix tren tap TEST (Accuracy: {test_acc*100:.1f}%)', fontsize=12, fontweight='bold', pad=12)
plt.xlabel('Nhan Du Doan (Predicted)', fontsize=11)
plt.ylabel('Nhan Thuc Te (Ground Truth)', fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
cm_path = os.path.join(OUTPUT_DIR, 'confusion_matrix.png')
plt.tight_layout()
plt.savefig(cm_path, dpi=300)
plt.show()
print(f'[INFO] Da luu ma tran nham lan vao: {cm_path}')

# 4. BAO CAO RIENG CHO LOP 'none' (KHONG PHAI TIEN)
none_idx = class_to_idx.get('none')
none_metrics_report = {}

if none_idx is not None:
    none_dict = report_dict.get('none', {})
    none_prec = none_dict.get('precision', 0.0)
    none_rec  = none_dict.get('recall', 0.0)
    none_f1   = none_dict.get('f1-score', 0.0)
    
    # Ti le anh tien that bi doan nham thanh none (Tu choi tien that)
    mask_money = (y_true != none_idx)
    total_money = int(np.sum(mask_money))
    money_as_none_count = int(np.sum(mask_money & (y_pred == none_idx)))
    money_as_none_rate = (money_as_none_count / total_money * 100.0) if total_money > 0 else 0.0
    
    # Ti le anh none bi doan nham thanh menh gia tien (Doc bua nguy hiem)
    mask_none = (y_true == none_idx)
    total_none = int(np.sum(mask_none))
    none_as_money_count = int(np.sum(mask_none & (y_pred != none_idx)))
    none_as_money_rate = (none_as_money_count / total_none * 100.0) if total_none > 0 else 0.0
    
    print('=' * 60)
    print('BAO CAO CHUYEN BIET CHO LOP "NONE" (KHONG PHAI TIEN):')
    print(f'  - Precision cua lop none: {none_prec * 100:.2f}%')
    print(f'  - Recall cua lop none:    {none_rec * 100:.2f}%')
    print(f'  - F1-Score cua lop none:  {none_f1 * 100:.2f}%')
    print('-' * 60)
    print(f'  - TI LE ANH TIEN THAT BI DOAN NHAM THANH NONE (Tu choi to tien that):')
    print(f'    + So luong anh bi nham: {money_as_none_count} / {total_money} anh tien')
    print(f'    + Ti le phan tram:     {money_as_none_rate:.2f}%')
    print(f'    + Tac dong thuc te:    Lam app khong nhan dien to tien that cua nguoi khiem thi.')
    print('-' * 60)
    print(f'  - TI LE ANH NONE BI DOAN NHAM THANH MENH GIA TIEN (Doc bua):')
    print(f'    + So luong anh bi nham: {none_as_money_count} / {total_none} anh none')
    print(f'    + Ti le phan tram:     {none_as_money_rate:.2f}%')
    print(f'    + Tac dong thuc te:    Rat nguy hiem! App nhan nham do vat xung quanh thanh tien.')
    print('=' * 60)
    
    none_metrics_report = {
        'precision': float(none_prec),
        'recall': float(none_rec),
        'f1_score': float(none_f1),
        'money_misclassified_as_none_count': int(money_as_none_count),
        'money_misclassified_as_none_rate_pct': float(money_as_none_rate),
        'none_misclassified_as_money_count': int(none_as_money_count),
        'none_misclassified_as_money_rate_pct': float(none_as_money_rate)
    }

## Bước 10: Xuất mô hình TensorFlow Lite (Lượng tử hóa FP16), Nhãn & Metrics
- **Chuyển đổi sang TensorFlow Lite FP16:** Giảm dung lượng file từ ~16MB xuống ~8MB, tăng tốc suy luận trên NPU/GPU điện thoại Android mà hầu như không suy giảm độ chính xác.
- **File `currency_labels.txt`:** Lưu danh sách nhãn lớp theo đúng thứ tự index softmax (có nhãn `none`).
- **File `label_display_vi.json`:** Ánh xạ tên nhãn sang câu đọc tiếng Việt, trong đó **nhãn `none` được gán giá trị chuỗi rỗng (``)** để ứng dụng không đọc mệnh giá khi không có tiền.
- **File `metrics.json`:** Lưu đầy đủ kết quả thực nghiệm, bao gồm cả các chỉ số chuyên biệt của lớp `none`.

In [ ]:
# @title Xuat TFLite FP16, currency_labels.txt (co 'none'), label_display_vi.json va metrics.json

TFLITE_PATH = os.path.join(OUTPUT_DIR, 'currency_mobilenetv3.tflite')
LABELS_PATH = os.path.join(OUTPUT_DIR, 'currency_labels.txt')
LABEL_DISPLAY_PATH = os.path.join(OUTPUT_DIR, 'label_display_vi.json')
METRICS_JSON_PATH = os.path.join(OUTPUT_DIR, 'metrics.json')

# 1. Chuyen doi sang TFLite FP16
print('[INFO] Dang chuyen doi Keras model sang TensorFlow Lite FP16...')
converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite_bytes = converter.convert()

with open(TFLITE_PATH, 'wb') as f:
    f.write(tflite_bytes)

tflite_size_mb = os.path.getsize(TFLITE_PATH) / (1024 * 1024)
print(f'[SUCCESS] Xuat file TFLite thanh cong: {TFLITE_PATH} ({tflite_size_mb:.2f} MB)')

# 2. Luu file currency_labels.txt (co lop none)
with open(LABELS_PATH, 'w', encoding='utf-8') as f:
    for cname in class_names:
        f.write(f'{cname}\n')
print(f'[SUCCESS] Da luu {len(class_names)} nhan vao: {LABELS_PATH}')

# 3. Anh xa cau doc tieng Viet (none la chuoi rong theo yeu cau)
LABEL_DISPLAY_DEFAULT = {
    'none': '',  # Chuoi rong: khong phai tien thi khong doc ten menh gia
    '1k': 'một nghìn đồng',
    '2k': 'hai nghìn đồng',
    '5k': 'năm nghìn đồng',
    '10k': 'mười nghìn đồng',
    '20k': 'hai mươi nghìn đồng',
    '50k': 'năm mươi nghìn đồng',
    '100k': 'một trăm nghìn đồng',
    '200k': 'hai trăm nghìn đồng',
    '500k': 'năm trăm nghìn đồng',
    '200': 'hai trăm đồng',
    '500': 'năm trăm đồng',
    '1000': 'một nghìn đồng',
    '2000': 'hai nghìn đồng',
    '5000': 'năm nghìn đồng',
    '10000': 'mười nghìn đồng',
    '20000': 'hai mươi nghìn đồng',
    '50000': 'năm mươi nghìn đồng',
    '100000': 'một trăm nghìn đồng',
    '200000': 'hai trăm nghìn đồng',
    '500000': 'năm trăm nghìn đồng',
}

label_display_map = {}
print('=' * 60)
print('BANG DOI CHIEU NHAN -> GIONG DOC TIENG VIET:')
for cname in class_names:
    spoken_vi = LABEL_DISPLAY_DEFAULT.get(cname.lower(), f'{cname} đồng')
    label_display_map[cname] = spoken_vi
    display_text = f'"{spoken_vi}"' if spoken_vi else '"" (Chuỗi rỗng)'
    print(f'  - Lop [{cname:6s}]  ->  Doc: {display_text}')
print('=' * 60)

with open(LABEL_DISPLAY_PATH, 'w', encoding='utf-8') as f:
    json.dump(label_display_map, f, ensure_ascii=False, indent=2)
print(f'[SUCCESS] Da luu tu dien doc tieng Viet vao: {LABEL_DISPLAY_PATH}')

# 4. Kiem tra tinh nhat quan voi TFLite Interpreter
interpreter = tf.lite.Interpreter(model_path=TFLITE_PATH)
interpreter.allocate_tensors()
in_details = interpreter.get_input_details()
out_details = interpreter.get_output_details()
print('[INFO] Thong so TFLite Tensor:')
print(f'  - Input:  {in_details[0]["shape"]} | dtype: {in_details[0]["dtype"]}')
print(f'  - Output: {out_details[0]["shape"]} | dtype: {out_details[0]["dtype"]}')

# Danh gia tren tap Val de luu metrics
val_eval = best_model.evaluate(val_ds, verbose=0)
val_acc = float(val_eval[1])

# 5. Luu toan bo metrics thuc nghiem vao metrics.json
metrics_data = {
    'timestamp': datetime.now().isoformat(),
    'tensorflow_version': tf.__version__,
    'keras_version': keras.__version__,
    'random_seed': RANDOM_SEED,
    'num_classes': num_classes,
    'class_names': class_names,
    'samples_count': {
        'total': total_imgs,
        'unique_groups': num_unique_groups,
        'train': len(train_paths),
        'val': len(val_paths),
        'test': len(test_paths),
        'none_total': none_total,
        'none_train': none_train_count,
        'none_val': none_val_count,
        'none_test': none_test_count,
        'none_percentage': float(none_percentage)
    },
    'accuracy': {
        'val_accuracy': float(val_acc),
        'test_accuracy': float(test_acc),
    },
    'none_class_metrics': none_metrics_report,
    'classification_report': report_dict
}

with open(METRICS_JSON_PATH, 'w', encoding='utf-8') as f:
    json.dump(metrics_data, f, ensure_ascii=False, indent=2)
print(f'[SUCCESS] Da luu toan bo metrics vao: {METRICS_JSON_PATH}')

## Hướng Dẫn Tích Hợp Mô Hình Vào Flutter App "Mắt Việt"

Khi đưa mô hình `currency_mobilenetv3.tflite` vào ứng dụng di động Flutter (`lib/features/currency/data/currency_classifier.dart`), cần tuân thủ đúng các thông số sau:

| Thuộc tính | Yêu cầu kỹ thuật | Ghi chú quan trọng |
| :--- | :--- | :--- |
| **Kích thước ảnh** | `224 x 224` pixel | Resize từ Camera Stream |
| **Không gian màu** | `RGB` (3 kênh) | Chuyển đổi từ `YUV420` của Camera |
| **Kiểu dữ liệu (dtype)** | `Float32List` | Tensor đầu vào `[1, 224, 224, 3]` |
| **Dải giá trị Pixel** | **$0.0 \rightarrow 255.0$** | **TUYỆT ĐỐI KHÔNG chia cho 255.0** (Mô hình đã nhúng sẵn lớp chuẩn hóa) |
| **Xử lý lớp `none`** | `label_display_map['none'] == ` | Khi giá trị đọc là rỗng ``, app **KHÔNG đọc tên mệnh giá** mà có thể thông báo: *"Không có tiền trong khung hình"* hoặc giữ im lặng |

In [ ]:
# @title Dong goi ket qua, sao luu vao Drive va tai xuong

# 1. Dong goi ZIP outputs
ZIP_OUTPUT_PATH = '/content/vision_assist_outputs.zip'
print('[INFO] Dang dong goi tep ZIP...')
with zipfile.ZipFile(ZIP_OUTPUT_PATH, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, _, files in os.walk(OUTPUT_DIR):
        for f_name in files:
            f_p = os.path.join(root, f_name)
            rel_p = os.path.relpath(f_p, OUTPUT_DIR)
            zipf.write(f_p, arcname=rel_p)

print(f'[SUCCESS] Da tao file ZIP: {ZIP_OUTPUT_PATH} ({os.path.getsize(ZIP_OUTPUT_PATH)/(1024*1024):.2f} MB)')

# 2. Sao luu vao Google Drive
try:
    if os.path.exists('/content/drive/MyDrive'):
        os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
        for item in os.listdir(OUTPUT_DIR):
            src_item = os.path.join(OUTPUT_DIR, item)
            dst_item = os.path.join(DRIVE_OUTPUT_DIR, item)
            if os.path.isfile(src_item):
                shutil.copy2(src_item, dst_item)
        shutil.copy2(ZIP_OUTPUT_PATH, os.path.join(DRIVE_OUTPUT_DIR, 'vision_assist_outputs.zip'))
        print(f'[SUCCESS] DA SAO LUU THANH CONG VAO GOOGLE DRIVE: {DRIVE_OUTPUT_DIR}')
    else:
        print('[INFO] Google Drive chua duoc gan ket, bo qua buoc sao luu Drive.')
except Exception as e:
    print(f'[WARNING] Khong the sao luu len Drive: {e}')

# 3. In Bang tong ket
print('=' * 60)
print('BANG TONG KET KET QUA HUAN LUYEN (SUMMARY):')
print('=' * 60)
summary_table = [
    ('Kien truc mang', 'MobileNetV3-Large (Transfer Learning)'),
    ('So luong lop', f'{num_classes} lop (Bao gom lop none)'),
    ('Tong so anh', f'{total_imgs} anh ({num_unique_groups} nhom sau Deduplication)'),
    ('Ty le lop none', f'{none_percentage:.2f}% ({none_total} anh)'),
    ('Tap phan chia', f'Train: {len(train_paths)} | Val: {len(val_paths)} | Test: {len(test_paths)}'),
    ('Validation Accuracy', f'{val_acc * 100:.2f}%'),
    ('Test Accuracy', f'{test_acc * 100:.2f}%'),
    ('Precision lop none', f'{none_metrics_report.get("precision", 0)*100:.2f}%'),
    ('Recall lop none', f'{none_metrics_report.get("recall", 0)*100:.2f}%'),
    ('Tien bi doan nham thanh none', f'{none_metrics_report.get("money_misclassified_as_none_rate_pct", 0):.2f}%'),
    ('None bi doan nham thanh tien', f'{none_metrics_report.get("none_misclassified_as_money_rate_pct", 0):.2f}%'),
    ('Dinh dang xuat', f'TensorFlow Lite FP16 ({tflite_size_mb:.2f} MB)'),
]
for k, v in summary_table:
    print(f'  - {k:32s}: {v}')
print('=' * 60)

# 4. Tai file ve may
try:
    from google.colab import files
    print('[INFO] Dang tai cac tep ket qua ve may tinh cua ban...')
    files_to_download = [
        TFLITE_PATH,
        LABELS_PATH,
        LABEL_DISPLAY_PATH,
        METRICS_JSON_PATH,
        cm_path,
        curves_path,
        ZIP_OUTPUT_PATH
    ]
    for fp in files_to_download:
        if os.path.exists(fp):
            print(f'  - Tai xuong: {Path(fp).name}')
            files.download(fp)
except Exception as e:
    print(f'[INFO] Bo qua files.download: {e}')
    print(f'[INFO] Ban co the tai truc tiep file ZIP tai: {ZIP_OUTPUT_PATH}')